# AI Engineering MasterClass - Class 1 Lab
### Your first metered LLM call

**By the end of this notebook you will have:**

1. A working provider connection, verified.
2. A call that reports the tokens it used and what they cost.
3. A running spend total for the session.
4. A guard that refuses to make a call that would push you past your budget.
5. The same prompt priced across three different models, saved as a result file.

Nothing in this notebook costs you money. Everything runs on the provider's free tier,
against open-weight models. That is true for all twenty classes.

**Read before you run:** you learn nothing from pressing Shift+Enter on all cells.
Run one cell, read the output, predict what the next one will print, then run it.

---
## Step 0 - Environment check

Your key lives in a `.env` file in this folder. It never lives in the notebook,
and `.env` goes into `.gitignore` before your first commit.

A key pasted into a cell ends up in your git history, and git history is forever.

In [8]:
import os, json, time, textwrap
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

# Accept either spelling so the notebook runs on anyone's machine.
API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("Grok_API_KEY")

assert API_KEY, "No key found. Put GROQ_API_KEY=... in a .env file next to this notebook."
print("Key loaded.  starts with:", API_KEY[:4] + "...", " length:", len(API_KEY))
print("Never print more of a key than this.")

Key loaded.  starts with: gsk_...  length: 56
Never print more of a key than this.


### A note on the name

The key in this folder is a **Groq** key. It starts with `gsk_` and it talks to
`api.groq.com`.

**Groq** is an inference provider. It runs open-weight models (GPT-OSS, Qwen, Llama)
on its own hardware, very fast, with a free tier.

**Grok** is a model made by xAI. Its keys start with `xai-` and it talks to `api.x.ai`.

Two different companies, one letter apart. Confusing them is a rite of passage and
costs people an afternoon. Read the key prefix, not the brand name you remember.

In [9]:
from groq import Groq

client = Groq(api_key=API_KEY)
print("Client ready.")

Client ready.


---
## Step 1 - What can you actually call?

Before you pick a model, ask the provider what it has, what each one's context window is,
and what it charges. Do not trust a blog post, a YouTube video, or me. Ask the API.

Prices and model names change every few weeks. The habit of asking does not.

In [10]:
import requests

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
resp.raise_for_status()
models = resp.json()["data"]

# Build a lookup of live prices, in dollars per million tokens.
PRICES = {}
for m in models:
    p = m.get("pricing") or {}
    if p.get("prompt") and p.get("completion"):      # text models only
        PRICES[m["id"]] = {
            "in":  float(p["prompt"]) * 1_000_000,
            "out": float(p["completion"]) * 1_000_000,
        }

print(f"{'model':38} {'context':>9} {'$/M in':>9} {'$/M out':>9}")
print("-" * 68)
for m in sorted(models, key=lambda x: x["id"]):
    pr = PRICES.get(m["id"])
    if not pr:
        continue
    print(f"{m['id']:38} {m.get('context_window', 0):>9,} {pr['in']:>9.3f} {pr['out']:>9.3f}")

model                                    context    $/M in   $/M out
--------------------------------------------------------------------
meta-llama/llama-prompt-guard-2-22m          512     0.030     0.030
meta-llama/llama-prompt-guard-2-86m          512     0.040     0.040
openai/gpt-oss-120b                      131,072     0.150     0.600
openai/gpt-oss-20b                       131,072     0.075     0.300
openai/gpt-oss-safeguard-20b             131,072     0.075     0.300
qwen/qwen3.8-27b                         131,072     0.800     4.000


**Look at that table before you move on.**

- Output costs more than input on every single model. Usually four times more.
- The spread between the cheapest and the dearest is more than ten times.
- Context windows are large now. A large window is not an invitation to fill it. You pay for every token in it.

Pick your working model for this class:

In [11]:
MODEL = "openai/gpt-oss-20b"   # cheapest capable text model on this account today

assert MODEL in PRICES, f"{MODEL} is not available on this key. Pick one from the table above."
RATE_IN, RATE_OUT = PRICES[MODEL]["in"], PRICES[MODEL]["out"]
print(f"Using {MODEL}")
print(f"  input  ${RATE_IN:.3f} per million tokens")
print(f"  output ${RATE_OUT:.3f} per million tokens")

Using openai/gpt-oss-20b
  input  $0.075 per million tokens
  output $0.300 per million tokens


---
## Step 2 - Your first call

One prompt, one answer. Nothing clever yet.

In [12]:
PROMPT = "Explain what a token is in a large language model, in three sentences, for a beginner."

r = client.chat.completions.create(
    model=MODEL,
    max_tokens=512,
    temperature=0.3,
    messages=[
        {"role": "system", "content": "You are a precise technical teacher. No filler."},
        {"role": "user", "content": PROMPT},
    ],
)

print(textwrap.fill(r.choices[0].message.content, 88))

A token is the smallest unit of text that a language model processes, such as a word,
sub‑word, or punctuation mark. The model learns by predicting the next token in a
sequence, using embeddings that map each token to a high‑dimensional vector. During
training and inference, tokens are converted to and from these vectors, enabling the
model to generate coherent text.


---
## Step 3 - Count it and price it

That call just cost money. Someone paid for it. In this course you will always know
how much, because you cannot engineer what you do not measure.

Three numbers come back on every response: input tokens, output tokens, and the time it took.

In [13]:
class Spend:
    """Tracks what this session has cost so far."""
    def __init__(self):
        self.total = 0.0
        self.calls = 0

    def record(self, usage, rate_in, rate_out):
        cost = usage.prompt_tokens / 1e6 * rate_in + usage.completion_tokens / 1e6 * rate_out
        self.total += cost
        self.calls += 1
        return cost

spend = Spend()

u = r.usage
cost = spend.record(u, RATE_IN, RATE_OUT)

print(f"input tokens   {u.prompt_tokens}")
print(f"output tokens  {u.completion_tokens}")
print(f"total tokens   {u.total_tokens}")
print(f"cost this call ${cost:.8f}")
print(f"session total  ${spend.total:.8f}  over {spend.calls} call(s)")

input tokens   103
output tokens  241
total tokens   344
cost this call $0.00008002
session total  $0.00008002  over 1 call(s)


Those numbers are tiny. That is the trap.

A tiny number times a user base is a real bill. Do the multiplication now, while it is cheap
to learn the habit:

In [14]:
for n in (1_000, 100_000, 1_000_000):
    print(f"{n:>10,} calls like that one  ->  ${cost * n:,.2f}")

     1,000 calls like that one  ->  $0.08
   100,000 calls like that one  ->  $8.00
 1,000,000 calls like that one  ->  $80.02


---
## Step 4 - The guard that refuses to spend

A counter tells you what you spent. A guard stops you spending it.

Estimate **before** the call, not after. After is a bill.

The estimate does not need to be exact. It needs to be conservative: assume the model
writes the full `max_tokens` you allowed it, because sometimes it will.

In [15]:
BUDGET = 0.05   # dollars, for this whole notebook session


def estimate_cost(messages, max_out, rate_in, rate_out):
    """Conservative estimate. 4 characters per token is a rough but safe rule for English."""
    chars = sum(len(m["content"]) for m in messages)
    est_in = chars / 4
    return est_in / 1e6 * rate_in + max_out / 1e6 * rate_out


def guarded_call(messages, max_out=512, model=MODEL, **kw):
    rate_in, rate_out = PRICES[model]["in"], PRICES[model]["out"]
    est = estimate_cost(messages, max_out, rate_in, rate_out)

    if spend.total + est > BUDGET:
        raise RuntimeError(
            f"Refused. Would exceed budget: spent ${spend.total:.6f} "
            f"+ estimated ${est:.6f} > cap ${BUDGET:.4f}"
        )

    t0 = time.time()
    resp = client.chat.completions.create(
        model=model, max_tokens=max_out, temperature=0.3, messages=messages, **kw
    )
    took = time.time() - t0
    actual = spend.record(resp.usage, rate_in, rate_out)
    choice = resp.choices[0]

    return resp, {
        "model": model,
        "in_tokens": resp.usage.prompt_tokens,
        "out_tokens": resp.usage.completion_tokens,
        "finish_reason": choice.finish_reason,          # never ignore this
        "content_chars": len(choice.message.content or ""),
        "estimated": round(est, 8),
        "actual": round(actual, 8),
        "seconds": round(took, 2),
    }

In [16]:
msgs = [{"role": "user", "content": "Name three failure modes of retrieval augmented generation. One line each."}]
resp, log = guarded_call(msgs, max_out=300)

print(textwrap.fill(resp.choices[0].message.content, 88))
print()
print(json.dumps(log, indent=2))
print(f"\nsession total ${spend.total:.8f} of ${BUDGET} cap")



{
  "model": "openai/gpt-oss-20b",
  "in_tokens": 84,
  "out_tokens": 300,
  "finish_reason": "length",
  "content_chars": 0,
  "estimated": 9.139e-05,
  "actual": 9.63e-05,
  "seconds": 0.73
}

session total $0.00017632 of $0.05 cap


### Read that output again

The answer printed above is **empty**, and `finish_reason` says `length`.

You were charged for 300 output tokens and you received nothing.

This model is a **reasoning model**. Before it answers you it thinks, and those hidden
thinking tokens bill as output at the full output rate. It spent your entire 300 token
allowance thinking, hit the ceiling, and never got to the answer.

This is the single most common way beginners waste money on reasoning models, and the
failure is silent. Your code gets back an empty string, not an error.

**Rule: check `finish_reason` on every call.** `stop` means it finished. `length` means
you cut it off and probably paid for nothing.

In [17]:
for effort in ("low", "medium", "high"):
    resp, log = guarded_call(msgs, max_out=300, reasoning_effort=effort)
    answered = "yes" if log["content_chars"] else "NO ANSWER"
    print(f"effort={effort:7} out_tokens={log['out_tokens']:>4}  "
          f"finish={log['finish_reason']:<7} answered={answered:<10} ${log['actual']:.8f}")

effort=low     out_tokens= 117  finish=stop    answered=yes        $0.00004140
effort=medium  out_tokens= 180  finish=stop    answered=yes        $0.00006030
effort=high    out_tokens= 300  finish=length  answered=NO ANSWER  $0.00009630


Same model, same prompt, same budget cap. The only thing that changed was the dial.

At `low` it answers and costs you a fraction. At `high` it burns the ceiling and returns
an empty string at full price.

**The thinking budget is a dial, not a switch.** Turning it up is the most expensive thing
you can do by accident. Turn it up when the task genuinely needs multi-step reasoning,
and leave it down for classification, extraction and formatting.

We come back to this properly in Class 3, when you choose models for real.

### Make it refuse

A guard you have never seen trigger is a guard you do not know works.
Drop the cap below what you have already spent and watch it refuse.

In [18]:
_real_budget = BUDGET
BUDGET = 0.0000001          # deliberately absurd

try:
    guarded_call([{"role": "user", "content": "This call should never happen."}])
    print("PROBLEM: the guard did not fire. Go and find out why.")
except RuntimeError as e:
    print("Guard fired correctly:")
    print(" ", e)
finally:
    BUDGET = _real_budget
    print(f"\nBudget restored to ${BUDGET}")

Guard fired correctly:
  Refused. Would exceed budget: spent $0.000374 + estimated $0.000154 > cap $0.0000

Budget restored to $0.05


**Two habits you keep for the rest of this course:**

1. Set the cap in code, as you just did.
2. Set a hard spend limit in the provider dashboard as well.

Code you can edit by accident at 2am. The dashboard limit you cannot.

---
## Step 5 - The assignment

Same prompt. Three models. Measure what changes.

This is the whole job in miniature: you are not picking the best model, you are picking
the cheapest model that is good enough for this specific task. You cannot do that
without numbers.

In [19]:
CANDIDATES = ["openai/gpt-oss-20b", "openai/gpt-oss-120b", "qwen/qwen3.8-27b"]
CANDIDATES = [m for m in CANDIDATES if m in PRICES]

TASK = [{
    "role": "user",
    "content": "A user writes: 'my order never arrived and nobody is answering'. "
               "Classify the intent in one word, rate urgency 1-5, and draft a one sentence reply. "
               "Answer as JSON with keys intent, urgency, reply.",
}]

results = []
for m in CANDIDATES:
    try:
        # effort="low" on the reasoning models, for the reason you just saw
        kw = {"reasoning_effort": "low"} if m.startswith("openai/") else {}
        resp, log = guarded_call(TASK, max_out=400, model=m, **kw)
        log["output"] = resp.choices[0].message.content.strip()
        results.append(log)
        print(f"--- {m}  ({log['seconds']}s, ${log['actual']:.8f})")
        print(textwrap.fill(log["output"], 88))
        print()
    except Exception as e:
        print(f"--- {m}  FAILED: {e}\n")

--- openai/gpt-oss-20b  (0.47s, $0.00003285)
{"intent":"delivery_issue","urgency":5,"reply":"I’m sorry your order hasn’t
arrived—could you please provide your order number so we can investigate and resolve
this promptly?"}

--- openai/gpt-oss-120b  (0.66s, $0.00007410)
{   "intent": "delivery",   "urgency": 5,   "reply": "We're sorry to hear your order
hasn't arrived; we'll investigate immediately and get back to you as soon as possible."
}

--- qwen/qwen3.8-27b  (0.23s, $0.00023120)
{   "intent": "complaint",   "urgency": 4,   "reply": "We apologize for the
inconvenience and are immediately investigating your missing order to provide a swift
resolution." }



In [20]:
print(f"{'model':26} {'in':>5} {'out':>5} {'finish':>8} {'secs':>6} {'cost':>12} {'x cheapest':>11}")
print("-" * 80)
cheapest = min(r["actual"] for r in results) if results else 1
for r in sorted(results, key=lambda x: x["actual"]):
    print(f"{r['model']:26} {r['in_tokens']:>5} {r['out_tokens']:>5} "
          f"{r['finish_reason']:>8} {r['seconds']:>6} ${r['actual']:>11.8f} "
          f"{r['actual']/cheapest:>10.1f}x")

print(f"\nWhole notebook cost: ${spend.total:.8f} over {spend.calls} calls.")

model                         in   out   finish   secs         cost  x cheapest
--------------------------------------------------------------------------------
openai/gpt-oss-20b           118    80     stop   0.47 $ 0.00003285        1.0x
openai/gpt-oss-120b          118    94     stop   0.66 $ 0.00007410        2.3x
qwen/qwen3.8-27b              59    46     stop   0.23 $ 0.00023120        7.0x

Whole notebook cost: $0.00071247 over 8 calls.


In [21]:
out = Path.cwd() / "class1_results.json"
out.write_text(json.dumps({
    "model_used": MODEL,
    "session_cost_usd": round(spend.total, 8),
    "calls": spend.calls,
    "comparison": results,
}, indent=2))
print("Saved", out)

Saved /Users/tahmid/Tahmid_AI_Engineer_MasterClass_Batch2/class1_results.json


---
## What you hand in before Class 2

Push a repository containing:

1. This notebook, run, with your outputs visible.
2. `class1_results.json`.
3. A `README.md` of three lines: which model you would put in production for that
   classification task, what it costs per thousand requests, and why you rejected the other two.

The README is the part that is actually marked. Anyone can run three calls.
The judgement is the deliverable.

**Next class:** Class 2, AI IDE Mastery. Cursor, Antigravity and agentic coding.
Come with this repo working.